In [0]:
-- Step 1, 2, 3은 이전과 동일
-- Step 1: 월별 활동 기록 추출
WITH user_monthly_activity AS (
    SELECT
        DISTINCT
        DATE_FORMAT(log_create_time, 'yyyy-MM') AS activity_month,
        device_name,
        mac_addr
    FROM kic_data_ods.tlamp.normal_log_webos24
    WHERE 1=1
        AND log_create_time >= '2024-04-01'
        AND DEVICE_NAME in ('k24', 'k8lpn2', 'o22n2', 'o24')
        AND X_Device_Country = 'KR'
        AND context_name = 'LSM'
        AND normal_log:app_id = 'com.webos.app.lgchannels'
        AND normal_log:visible = true
),
-- Step 2: 첫 활동 월(코호트) 정의
user_first_activity AS (
    SELECT
        device_name,
        mac_addr,
        MIN(activity_month) AS cohort_month
    FROM user_monthly_activity
    GROUP BY device_name, mac_addr
),
-- Step 3: 코호트 월로부터의 경과 월 계산
cohort_retention_data AS (
    SELECT
        fa.cohort_month,
        fa.device_name,
        fa.mac_addr,
        CAST(FLOOR(MONTHS_BETWEEN(
            TO_DATE(ma.activity_month, 'yyyy-MM'),
            TO_DATE(fa.cohort_month, 'yyyy-MM')
        )) AS INT) AS month_number
    FROM user_first_activity fa
    JOIN user_monthly_activity ma
      ON fa.mac_addr = ma.mac_addr AND fa.device_name = ma.device_name
    WHERE
        fa.cohort_month BETWEEN '2024-04' AND '2025-04'
),

-- Step 4: 코호트별, 월차별 잔존 유저 수(절대값)를 계산합니다. (중간 단계)
cohort_absolute_counts AS (
    SELECT
        cohort_month,
        device_name,
        COUNT(DISTINCT CASE WHEN month_number = 0 THEN mac_addr END) AS M0,
        COUNT(DISTINCT CASE WHEN month_number = 1 THEN mac_addr END) AS M1,
        COUNT(DISTINCT CASE WHEN month_number = 2 THEN mac_addr END) AS M2,
        COUNT(DISTINCT CASE WHEN month_number = 3 THEN mac_addr END) AS M3,
        COUNT(DISTINCT CASE WHEN month_number = 4 THEN mac_addr END) AS M4,
        COUNT(DISTINCT CASE WHEN month_number = 5 THEN mac_addr END) AS M5,
        COUNT(DISTINCT CASE WHEN month_number = 6 THEN mac_addr END) AS M6,
        COUNT(DISTINCT CASE WHEN month_number = 7 THEN mac_addr END) AS M7,
        COUNT(DISTINCT CASE WHEN month_number = 8 THEN mac_addr END) AS M8,
        COUNT(DISTINCT CASE WHEN month_number = 9 THEN mac_addr END) AS M9,
        COUNT(DISTINCT CASE WHEN month_number = 10 THEN mac_addr END) AS M10,
        COUNT(DISTINCT CASE WHEN month_number = 11 THEN mac_addr END) AS M11,
        COUNT(DISTINCT CASE WHEN month_number = 12 THEN mac_addr END) AS M12
    FROM cohort_retention_data
    GROUP BY cohort_month, device_name
)

-- Step 5: 절대값 카운트를 바탕으로 잔존율(%)을 계산하여 최종 결과를 출력합니다.
SELECT
    cohort_month,
    device_name,
    M0 AS initial_users,
    M1 AS M1_retained_users,
    M2 AS M2_retained_users,
    M3 AS M3_retained_users,
    M4 AS M4_retained_users,
    M5 AS M5_retained_users,
    M6 AS M6_retained_users,
    M7 AS M7_retained_users,
    M8 AS M8_retained_users,
    M9 AS M9_retained_users,
    M10 AS M10_retained_users,
    M11 AS M11_retained_users,
    M12 AS M12_retained_users,    

    100 AS intial_rate,
        -- M+1 잔존율 및 절대값
    ROUND(M1 / M0 * 100, 2) AS M1_retention_rate,
    
    -- M+2 잔존율 및 절대값
    ROUND(M2 / M0 * 100, 2) AS M2_retention_rate,
    
    -- M+3 잔존율 및 절대값
    ROUND(M3 / M0 * 100, 2) AS M3_retention_rate,
    
    -- M+4 부터 M+12 까지 동일한 패턴으로 추가
    ROUND(M4 / M0 * 100, 2) AS M4_retention_rate,
    ROUND(M5 / M0 * 100, 2) AS M5_retention_rate,
    ROUND(M6 / M0 * 100, 2) AS M6_retention_rate,
    ROUND(M7 / M0 * 100, 2) AS M7_retention_rate,
    ROUND(M8 / M0 * 100, 2) AS M8_retention_rate,
    ROUND(M9 / M0 * 100, 2) AS M9_retention_rate,
    ROUND(M10 / M0 * 100, 2) AS M10_retention_rate,
    ROUND(M11 / M0 * 100, 2) AS M11_retention_rate,
    ROUND(M12 / M0 * 100, 2) AS M12_retention_rate


FROM cohort_absolute_counts
-- 0으로 나누는 오류 방지 (신규 유저가 없는 경우 해당 행 제외)
WHERE M0 > 0
ORDER BY cohort_month, device_name;